In [3]:
import pandas as pd

In [4]:
master_table = pd.read_csv('master_table.csv')

In [6]:
master_table.shape

(119143, 40)

In [10]:
master_table['order_purchase_timestamp'] = pd.to_datetime(master_table['order_purchase_timestamp'])
master_table['order_approved_at'] = pd.to_datetime(master_table['order_approved_at'])
master_table['order_delivered_carrier_date'] = pd.to_datetime(master_table['order_delivered_carrier_date'])
master_table['order_delivered_customer_date'] = pd.to_datetime(master_table['order_delivered_customer_date'])
master_table['order_estimated_delivery_date'] = pd.to_datetime(master_table['order_estimated_delivery_date'])
master_table['shipping_limit_date'] = pd.to_datetime(master_table['shipping_limit_date'])
master_table['review_answer_timestamp']= pd.to_datetime(master_table['review_answer_timestamp'])

## Feature Engineering

#### Business question: How many days does it take to deliver an order after purchase?

In [13]:
master_table['delivery_days'] = (master_table['order_delivered_customer_date'] - master_table['order_purchase_timestamp']).dt.days

#### Business question: How quickly are orders approved after being placed?

In [16]:
master_table['approval_hours'] = (master_table['order_approved_at'] - master_table['order_purchase_timestamp']).dt.total_seconds()/3600

#### Business question: How long does it take the seller to hand the package over to the carrier after approval?

In [17]:
master_table['shipping_days'] = (master_table['order_delivered_carrier_date'] - master_table['order_approved_at']).dt.days

#### Business question: Once the carrier receives the package, how many days does it take to reach the customer?

In [18]:
master_table['carrier_to_customer_days'] = (master_table['order_delivered_customer_date'] - master_table['order_delivered_carrier_date']).dt.days

In [21]:
master_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_comment_message,review_creation_date,review_answer_timestamp,seller_zip_code_prefix,seller_city,seller_state,delivery_days,approval_hours,shipping_days,carrier_to_customer_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,9350.0,maua,SP,8.0,0.178333,2.0,6.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,9350.0,maua,SP,8.0,0.178333,2.0,6.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,9350.0,maua,SP,8.0,0.178333,2.0,6.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,31570.0,belo horizonte,SP,13.0,30.713889,0.0,12.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,NaN,2018-08-18,2018-08-22 19:07:58,14840.0,guariba,SP,9.0,0.276111,0.0,9.0


#### Time Intelligence Feature

In [23]:
master_table['purchase_year'] = master_table['order_purchase_timestamp'].dt.year

In [24]:
master_table['purchase_month'] = master_table['order_purchase_timestamp'].dt.month

In [25]:
master_table['purchase_month_name'] = master_table['order_purchase_timestamp'].dt.month_name()

In [26]:
master_table['purchase_day'] = master_table['order_purchase_timestamp'].dt.day_name()

In [27]:
master_table['purchase_quarter'] = master_table['order_purchase_timestamp'].dt.quarter

In [28]:
master_table['is_weekend'] = master_table['order_purchase_timestamp'].dt.dayofweek>=5

#### Review Feature

In [29]:
master_table['review_category'] = master_table['review_score'].map({1:'Negative',2:'Negative',3:'Neutral',4:'Positive',5:'Positive'})

#### Financial Feature

In [31]:
master_table['total_item_value'] = master_table['price']+master_table['freight_value']

In [33]:
master_table['freight_perc'] = (master_table['freight_value']/master_table['price'])*100

#### Flags

In [41]:
master_table['delivery_delay'] = (master_table['order_delivered_customer_date']- master_table['order_estimated_delivery_date']).dt.days

In [42]:
master_table['is_late_delivery'] = master_table['delivery_delay']>0

In [47]:
threshold = master_table['total_item_value'].quantile(0.75)
master_table['high_value_order'] = (master_table['total_item_value']>=threshold)

In [50]:
master_table['order_value_category'] = pd.qcut(master_table['total_item_value'], q=4,labels=['low','medium','high','premium'])

In [52]:
master_table.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,purchase_day,purchase_quarter,is_weekend,review_category,total_item_value,freight_perc,delivery_delay,is_late_delivery,high_value_order,order_value_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,Monday,4,False,Positive,38.71,29.076359,-8.0,False,False,low
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,Monday,4,False,Positive,38.71,29.076359,-8.0,False,False,low
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,Monday,4,False,Positive,38.71,29.076359,-8.0,False,False,low
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,Tuesday,3,False,Positive,141.46,19.174389,-6.0,False,False,high
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,Wednesday,3,False,Positive,179.12,12.020013,-18.0,False,True,premium
5,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,Saturday,4,True,Positive,72.20,60.444444,-13.0,False,False,medium
6,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,Tuesday,1,False,Positive,28.62,43.819095,-10.0,False,False,low
7,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,80bb27c7c16e8f973207a5086ab329e2,86320,...,Sunday,3,True,Positive,175.26,18.498986,-6.0,False,True,premium
8,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,36edbb3fb164b1f16485364b6fb04c73,98900,...,Tuesday,2,False,Negative,65.95,32.164329,NaN,False,False,medium
9,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,932afa1e708222e5821dac9cd5db4cae,26525,...,Tuesday,2,False,Positive,75.16,25.287548,-12.0,False,False,medium


In [53]:
master_table.to_pickle('master_table.pkl')